<a href="https://colab.research.google.com/github/LGF92/tuscan-whisper-eval/blob/main/tuscan_whisper_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datacollective
import os
from google.colab import userdata
os.environ["MDC_API_KEY"] = userdata.get("MDC_API_KEY")

from datacollective import download_dataset
path = download_dataset("cmu5xagm300dyo1075qdhf20m")
print(path)

  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datacollective]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
█████████████████████████████████████████████████🦊 100.0% (9.8 GB/9.8 GB) Average: 64.3 MB/s Total time: 02:36
/root/.mozdata/datasets/common-voice-scripted-speech-27-0-italia-5ca1c1b9.tar.gz


In [2]:
!ls -lh {path}

-rw-r--r-- 1 root root 9.9G Nov 14 15:09 /root/.mozdata/datasets/common-voice-scripted-speech-27-0-italia-5ca1c1b9.tar.gz


In [3]:
!mkdir -p /content/cv
!tar -xzf {path} -C /content/cv --wildcards "*.tsv"

In [4]:
!find /content/cv -name "*.tsv"

/content/cv/cv-corpus-27.0-2026-09-11/it/invalidated.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/dev.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/validated_sentences.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/reported.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/validated.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/other.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/test.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/clip_durations.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/unvalidated_sentences.tsv
/content/cv/cv-corpus-27.0-2026-09-11/it/train.tsv


In [5]:
import pandas as pd, glob
tsv = glob.glob("/content/cv/**/validated.tsv", recursive=True)[0]
df = pd.read_csv(tsv, sep="\t", low_memory=False)
print(len(df))
print(df.columns.tolist())

240384
['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


In [6]:
pattern = "tosc|tuscan|firenz|fiorent|pisa|pisan|sien|senese|livorn|lucc|arezz|aretin|massa|masses|carrar|grosset|prato|prates|pistoi|empol"
tusc = df[df["accents"].str.contains(pattern, case=False, na=False)]
print(len(tusc), tusc["client_id"].nunique())
print(tusc["accents"].value_counts().head(20))

561 13
accents
Tosco-Emiliano                                     170
Toscano|Sud senese                                 131
Toscano|viareggino                                  66
Toscano                                             59
Fiorentino                                          54
Possibile intercalare toscano                       43
Pulito con leggera inflessione toscana e romana     30
Toscano|Fiorentino|chianti                           5
Inglese|pisano                                       3
Name: count, dtype: int64


In [7]:
drop = "Tosco-Emiliano|Inglese"
tusc = tusc[~tusc["accents"].str.contains(drop, case=False, na=False)]
print(len(tusc), tusc["client_id"].nunique())

388 11


In [8]:
print(tusc.groupby("client_id").size().sort_values(ascending=False))
print(tusc.drop_duplicates("client_id")["gender"].value_counts())

client_id
2f030347047a860ef0b12eca611ed64423cf3e0cdee862e5bbbf08be50fcbbde8b35d8b47f8489d0e1eecce3a5c8118e672b4ad4a2c2849602a854877aa14d61    131
508c467018c8fb4a74d2e512f5606ace8882b079875a8f4765922421de1deb5f77e2789be65ccf97435f3de8357ccabe8df2aea55025e2a9c1a4d207badaf4c9     61
3e8b30da160846e76a9092de5ef72de1899ae1270b7aba607ef22fa9b74a5a3e601ae98a782047bc8090b28cbcbbf3b40e9ea65b229c58d869a67ec583398224     54
5cbb833bb604fb5a7be7ab7131b9470b3efbdc9cd59f811516a6f7d0be00e780e5be5108b2e872cfe4097f489258b93b3786b84303187b456bf88dea6bb30ed4     43
4cf8a452f960e1659a35dfca461d90d8c9a8173841dcbfa27e8687290d8f9c9c956200e45443b37832a2d727591698d45d9e857a0db4c1891256dbe7aaaba39a     41
b25b6e25a9b544bfaee841cc9c01427d78e9349da6fd98ac8a95d75ccaa4dd822ef39eb297a33e025dedb4159a53e37a6d62c9ad88e628661425f45f0ddd5aad     30
6648d7188acfc5af19a7e59f590a14b5628ebeda6cd32db4e14be92d6b6e3acb492d5b3dc1a9f6fa94f49ec38ee214973471d8db074c03f778d9095c7a954c0b     14
252cc12366ea172a48716c4dadbc8b28105635

In [9]:
tusc = tusc.groupby("client_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 50), random_state=42))
print(len(tusc), tusc["client_id"].nunique())

292 11


/tmp/ipykernel_844/55140644.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tusc = tusc.groupby("client_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 50), random_state=42))


In [10]:
import os
base = os.path.dirname(tsv).replace("/content/cv/", "")
names = [f"{base}/clips/{p}" for p in tusc["path"]]
open("/content/want.txt", "w").write("\n".join(names))
!tar -xzf {path} -C /content/cv -T /content/want.txt

In [11]:
!find /content/cv -name "*.mp3" | wc -l

292


In [12]:
!cp /content/cv/{base}/clips/*.mp3 /content/tuscan_asr_clips/ 2>/dev/null || (mkdir -p /content/tuscan_asr_clips && cp /content/cv/{base}/clips/*.mp3 /content/tuscan_asr_clips/)
tusc.to_csv("/content/tuscan_asr_clips/tuscan_cv.csv", index=False)
!cd /content && zip -qr tuscan_asr.zip tuscan_asr_clips
from google.colab import files
files.download("/content/tuscan_asr.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
print(df["accents"].value_counts().head(40))

accents
trentino|lucano                                                                                                                                                                 5366
Tendente al siculo, ma non marcato                                                                                                                                              3165
Romanesco                                                                                                                                                                       2373
Veneto                                                                                                                                                                          1621
Piemontese                                                                                                                                                                       718
comune|neutrale                                                                        

In [14]:
std_labels = ["comune|neutrale", "Italiano con buona dizione"]
std = df[df["accents"].isin(std_labels)]
print(len(std), std["client_id"].nunique())
print(std.drop_duplicates("client_id")["gender"].value_counts())

304 2
gender
male_masculine    1
Name: count, dtype: int64


In [15]:
tusc_ids = set(tusc["client_id"])
labelled = df[df["accents"].notna() & ~df["client_id"].isin(tusc_ids)]
labelled = labelled[~labelled["accents"].str.contains(pattern + "|Inglese|Expat", case=False)]
counts = labelled["client_id"].value_counts()
speakers = counts[counts >= 25].sample(11, random_state=42).index
base_df = labelled[labelled["client_id"].isin(speakers)]
base_df = base_df.groupby("client_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 50), random_state=42))
print(len(base_df), base_df["client_id"].nunique())
print(base_df.drop_duplicates("client_id")[["accents", "gender"]])

475 11
                         accents           gender
44819      Madrelingua|Lombardia  female_feminine
59388                   Northern  female_feminine
61317   Romano|Italiano Standard   male_masculine
46184   Del Nord Italia|Lombardo              NaN
45402        Nord-Est (Emiliano)              NaN
85984                   Emiliano   male_masculine
101623      Italiano - Siciliano   male_masculine
95506                Meridionale   male_masculine
114314                Piemontese   male_masculine
151342                    Veneto   male_masculine
41428             neutro|monzese   male_masculine


/tmp/ipykernel_844/2390835680.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  base_df = base_df.groupby("client_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 50), random_state=42))


In [16]:
base_df = base_df[~base_df["gender"].str.contains("female", case=False, na=False)]
print(len(base_df), base_df["client_id"].nunique())

393 9


In [17]:
names = [f"{base}/clips/{p}" for p in base_df["path"]]
open("/content/want_base.txt", "w").write("\n".join(names))
!tar -xzf {path} -C /content/cv -T /content/want_base.txt
!find /content/cv -name "*.mp3" | wc -l

685


In [18]:
import shutil
os.makedirs("/content/baseline_clips", exist_ok=True)
for p in base_df["path"]:
    shutil.copy(f"/content/cv/{base}/clips/{p}", "/content/baseline_clips/")
base_df.to_csv("/content/baseline_clips/baseline_cv.csv", index=False)
!cd /content && zip -qr baseline_asr.zip baseline_clips
files.download("/content/baseline_asr.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!pip install -q openai-whisper
import whisper
model = whisper.load_model("turbo")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


100%|██████████████████████████████████████| 1.51G/1.51G [00:13<00:00, 123MiB/s]


In [20]:
import pandas as pd
tusc["group"] = "tuscan"
base_df["group"] = "baseline"
all_df = pd.concat([tusc, base_df], ignore_index=True)

hyps = []
for i, p in enumerate(all_df["path"]):
    r = model.transcribe(f"/content/cv/{base}/clips/{p}", language="it")
    hyps.append(r["text"].strip())
    if i % 50 == 0:
        print(i, r["text"])

all_df["whisper"] = hyps
all_df.to_csv("/content/results.csv", index=False)

0  Nel doppiaggio in italiano, Johnny diventa Jenny.
50  La cifra più significativa delle due verrà inviata per prima.
100  Microsoft dovrà correre a ripario al più presto se vorrà evitare di perdere la propria posizione.
150  La canzone è stata scritta e prodotta dal produttore Teddy Park.
200  L'album è dedicato al chitarrista Bruce Sonhouse.
250  La produzione cambiò portando l'opera lirica e il balletto all'interno del querino.
300  Questi soldati vengono inviati a distruggere l'esercito senza vita prima che possa raggiungere Idris.
350  Vive appena due o tre anni e non è gregario.
400  Cupi Avati propose i ruoli da protagonista a Roberto Benigni e Francesco Nuti.
450  Esistono molti tipi di bite con diverse indicazioni terapeutiche
500  scrive su due blog
550  Sono stati inoltre trovati frammenti di piante angiosperme monocotiledoni e angiosperme dicotiledoni.
600  In questa occasione una bomba causò danni leggeri alla nave.
650  Il documento riconosce tuttavia che tali insinuazio

In [21]:
!pip install -q jiwer
import jiwer, re

def norm(t):
    t = str(t).lower().replace("’", "'")
    t = re.sub(r"[^\w\s']", " ", t)
    return " ".join(t.split())

all_df["ref_n"] = all_df["sentence"].apply(norm)
all_df["hyp_n"] = all_df["whisper"].apply(norm)

for g, d in all_df.groupby("group"):
    print(g, round(jiwer.wer(list(d["ref_n"]), list(d["hyp_n"])) * 100, 2), "%")

baseline 4.19 %
tuscan 6.18 %


In [22]:
rows = []
for (g, s), d in all_df.groupby(["group", "client_id"]):
    w = jiwer.wer(list(d["ref_n"]), list(d["hyp_n"])) * 100
    rows.append((g, s[:8], len(d), round(w, 2)))
spk = pd.DataFrame(rows, columns=["group", "speaker", "clips", "wer"])
print(spk.sort_values(["group", "wer"], ascending=[True, False]).to_string(index=False))

   group  speaker  clips   wer
baseline c9ad761b     50  7.61
baseline 11d89dc6     50  5.64
baseline 4d288db5     50  4.98
baseline cf3caed0     26  3.60
baseline 4498061e     50  3.52
baseline 3caaf3d6     34  3.19
baseline 3e7de698     33  3.06
baseline 50c82af7     50  3.05
baseline 52fb6f9b     50  2.50
  tuscan 791889bc      5 25.00
  tuscan 252cc123      5 17.39
  tuscan 508c4670     50 10.35
  tuscan 3e8b30da     50  8.90
  tuscan 4cf8a452     41  5.21
  tuscan 6648d718     14  4.51
  tuscan 2f030347     50  3.90
  tuscan c0782407      3  3.85
  tuscan 5cbb833b     43  2.88
  tuscan b25b6e25     30  2.03
  tuscan a9d26f33      1  0.00


In [23]:
print(tusc.drop_duplicates("client_id")[["client_id", "accents"]].assign(client_id=lambda x: x["client_id"].str[:8]).to_string(index=False))

client_id                                         accents
 252cc123                      Toscano|Fiorentino|chianti
 2f030347                              Toscano|Sud senese
 3e8b30da                                      Fiorentino
 4cf8a452                                         Toscano
 508c4670                              Toscano|viareggino
 5cbb833b                   Possibile intercalare toscano
 6648d718                                         Toscano
 791889bc                              Toscano|viareggino
 a9d26f33                                         Toscano
 b25b6e25 Pulito con leggera inflessione toscana e romana
 c0782407                                         Toscano


In [24]:
V = "aeiouàèéìòóù"
def gorgia_flags(sentence):
    words = sentence.split()
    flags = []
    for j, w in enumerate(words):
        inside = re.search(f"[{V}](c[aouàòóù]|ch[ei]|t|p)[{V}h]", w) is not None
        prev_vowel = j > 0 and words[j-1][-1] in V
        start = re.match(f"(c[aouàòóù]|ch[ei]|t|p)[{V}h]", w) is not None
        flags.append(inside or (prev_vowel and start))
    return flags

stats = []
for _, row in all_df.iterrows():
    ref, hyp = row["ref_n"], row["hyp_n"]
    if not ref:
        continue
    flags = gorgia_flags(ref)
    wrong = [False] * len(flags)
    out = jiwer.process_words(ref, hyp)
    for ch in out.alignments[0]:
        if ch.type in ("substitute", "delete"):
            for k in range(ch.ref_start_idx, ch.ref_end_idx):
                wrong[k] = True
    for f, e in zip(flags, wrong):
        stats.append((row["group"], f, e))

st = pd.DataFrame(stats, columns=["group", "gorgia", "error"])
print(st.groupby(["group", "gorgia"])["error"].agg(["count", "mean"]).assign(mean=lambda x: (x["mean"]*100).round(2)))

                 count  mean
group    gorgia             
baseline False    3085  3.57
         True      710  3.66
tuscan   False    2360  5.89
         True      455  4.62


In [25]:
worst = all_df[all_df["client_id"].str[:8].isin(["508c4670", "3e8b30da"])]
worst = worst[worst["ref_n"] != worst["hyp_n"]]
for _, r in worst.iterrows():
    print("REF:", r["ref_n"])
    print("HYP:", r["hyp_n"])
    print()

REF: la ragazza si innamora di uno di loro che sembra ricambiare il sentimento
HYP: la ragazza si innamora di uno di loro che sembra di cambiare il sentimento

REF: il team digitale si farà anche carico di ingaggiare i rapporti con le pa coinvolte nella fase di sperimentazione
HYP: il team digitale si farà anche carico di ingaggiare da qualche colpia atto involte nella fase di sperimentazione

REF: mclaren et al j am
HYP: mclaren e al g a m

REF: i contrabbandieri
HYP: i contrabandieri

REF: e per giunta posti a due angoli opposti dell'esagono
HYP: e per giunta posti altoangoli opposti dell'esagono

REF: open range
HYP: r psych翻chnung

REF: il pubblico dominio è l ambito da cui estraiamo i mattoni con cui costruire la nostra cultura
HYP: il pubblico dominio è l'ambito da cui estraiamo i mattoni con cui costruire la nostra cultura

REF: ma anche zia emilia s'era avvicinata saettando il gruppo con lo sguardo
HYP: ma anche zia emilia si era avvicinata saettando il gruppo con lo sguardo

R

In [26]:
import unicodedata
def norm2(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())

all_df["ref_n"] = all_df["sentence"].apply(norm2)
all_df["hyp_n"] = all_df["whisper"].apply(norm2)

for g, d in all_df.groupby("group"):
    print(g, round(jiwer.wer(list(d["ref_n"]), list(d["hyp_n"])) * 100, 2), "%")

baseline 3.91 %
tuscan 5.69 %


In [27]:
def has_names(s):
    words = str(s).split()[1:]
    return any(w[:1].isupper() for w in words)

clean = all_df[~all_df["sentence"].apply(has_names)]
for g, d in clean.groupby("group"):
    print(g, len(d), round(jiwer.wer(list(d["ref_n"]), list(d["hyp_n"])) * 100, 2), "%")


baseline 236 3.01 %
tuscan 139 2.86 %


In [28]:
all_df["names"] = all_df["sentence"].apply(has_names)
for g, d in all_df.groupby("group"):
    n = d[d["names"]]
    print(g, "name sentences:", len(n), "of", len(d), "| WER on them:", round(jiwer.wer(list(n["ref_n"]), list(n["hyp_n"])) * 100, 2), "%")

baseline name sentences: 157 of 393 | WER on them: 5.15 %
tuscan name sentences: 153 of 292 | WER on them: 8.39 %


In [29]:
all_df.to_csv("/content/results.csv", index=False)
files.download("/content/results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
TRIG = {"a","e","o","da","fu","tre","che","se","ma","ha","ho","ne","chi","re","piu","puo","gia","cosi","perche","citta"}

def rs_flags(sentence):
    w = sentence.split()
    flags = [False] * len(w)
    for j in range(1, len(w)):
        if w[j-1] in TRIG and w[j][0] not in "aeiou":
            flags[j] = True
    return flags

stats = []
for _, row in all_df.iterrows():
    ref, hyp = row["ref_n"], row["hyp_n"]
    if not ref:
        continue
    flags = rs_flags(ref)
    wrong = [False] * len(flags)
    for ch in jiwer.process_words(ref, hyp).alignments[0]:
        if ch.type in ("substitute", "delete"):
            for k in range(ch.ref_start_idx, ch.ref_end_idx):
                wrong[k] = True
    for f, e in zip(flags, wrong):
        stats.append((row["group"], row["names"], f, e))

rs = pd.DataFrame(stats, columns=["group", "names", "raddoppiamento", "error"])
print(rs.groupby(["group", "raddoppiamento"])["error"].agg(["count", "mean"]).assign(mean=lambda x: (x["mean"]*100).round(2)))

                         count  mean
group    raddoppiamento             
baseline False            3568  3.34
         True              291  3.09
tuscan   False            2672  5.24
         True              191  5.24


In [31]:
!pip install -q transformers librosa
import librosa, torch
from transformers import pipeline

asr = pipeline("automatic-speech-recognition",
               model="jonatasgrosman/wav2vec2-large-xlsr-53-italian",
               device=0)

w2v = []
for i, p in enumerate(all_df["path"]):
    audio, _ = librosa.load(f"/content/cv/{base}/clips/{p}", sr=16000)
    w2v.append(asr(audio)["text"].strip())
    if i % 50 == 0:
        print(i, w2v[-1])

all_df["w2v"] = w2v
all_df["w2v_n"] = all_df["w2v"].apply(norm2)
for g, d in all_df.groupby("group"):
    print(g, round(jiwer.wer(list(d["ref_n"]), list(d["w2v_n"])) * 100, 2), "%")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

... (4 outputs left out)

[transformers] Could not load the `decoder` for jonatasgrosman/wav2vec2-large-xlsr-53-italian. Defaulting to raw CTC. Error: No module named 'kenlm'
[transformers] Try to install `kenlm`: `pip install kenlm
[transformers] Try to install `pyctcdecode`: `pip install pyctcdecode


0 nel doppiaggio in italiano joni diventa jenny


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


50 la scipa più simpognificativa delle due verrà inviata per prima
100 microsoft vov raccorrere a ripare al più presto se vorrà evitare di perdere la propria opposizione
150 la canzone è stata scritta e prodotto dal produttore teddy park
200 l'album è dedicato al chitarrista bruce son house
250 la produzione cambiò portando l'opera lidica di balletto all'interno del quirino
300 questi soldati vengono inviati a distruggere l'esercito senza vita prima che possa raggiungere idriss
350 vive appena due o tre anni e non è gregario
400 upi avati propose i ruoli da protagonista a roberto benigni e francesco nuti
450 esistono molti tipi di baight con diverse indicazioni terapeutiche
500 scrive su due blog
550 sono stati inoltre trovati frammenti di piante angeosperme monocotiledoni e angeosperme dicotiledoni
600 in questa occasione una bomba causò danni leggeri alla nave
650 il documento riconosce tuttavia che tali insinuazioni erano fondate
baseline 4.59 %
tuscan 8.1 %


In [32]:
def feature_test(hyp_col):
    rows = []
    for _, r in all_df.iterrows():
        ref, hyp = r["ref_n"], r[hyp_col]
        if not ref:
            continue
        g_flags = gorgia_flags(ref)
        r_flags = rs_flags(ref)
        wrong = [False] * len(g_flags)
        for ch in jiwer.process_words(ref, hyp).alignments[0]:
            if ch.type in ("substitute", "delete"):
                for k in range(ch.ref_start_idx, ch.ref_end_idx):
                    wrong[k] = True
        for g, rd, e in zip(g_flags, r_flags, wrong):
            rows.append((r["group"], g, rd, e))
    return pd.DataFrame(rows, columns=["group", "gorgia", "raddoppiamento", "error"])

for model_name, col in [("WHISPER", "hyp_n"), ("WAV2VEC2", "w2v_n")]:
    t = feature_test(col)
    print("=====", model_name, "=====")
    for feat in ["gorgia", "raddoppiamento"]:
        print(t.groupby(["group", feat])["error"].mean().mul(100).round(2), "\n")

===== WHISPER =====
group     gorgia
baseline  False     3.27
          True      3.52
tuscan    False     5.48
          True      3.95
Name: error, dtype: float64 

group     raddoppiamento
baseline  False             3.34
          True              3.09
tuscan    False             5.24
          True              5.24
Name: error, dtype: float64 

===== WAV2VEC2 =====
group     gorgia
baseline  False     4.35
          True      3.94
tuscan    False     7.73
          True      7.02
Name: error, dtype: float64 

group     raddoppiamento
baseline  False             4.32
          True              3.78
tuscan    False             7.63
          True              7.33
Name: error, dtype: float64 



In [33]:
wer_col = lambda d, col: round(jiwer.wer(list(d["ref_n"]), list(d[col])) * 100, 2)

for model_name, col in [("WHISPER", "hyp_n"), ("WAV2VEC2", "w2v_n")]:
    print("=====", model_name, "=====")
    for g, d in all_df.groupby("group"):
        print(g, "| no names:", wer_col(d[~d["names"]], col), "% | names:", wer_col(d[d["names"]], col), "%")

===== WHISPER =====
baseline | no names: 3.01 % | names: 5.15 %
tuscan | no names: 2.86 % | names: 8.39 %
===== WAV2VEC2 =====
baseline | no names: 3.32 % | names: 6.31 %
tuscan | no names: 5.23 % | names: 10.85 %
